# p-Laplace 2D — JAX + PETSc (MPI-parallel)

This notebook demonstrates the **JAX + PETSc DOF-partitioned** solver for the 2D p-Laplacian problem.

**Key features:**
- **JAX** automatic differentiation for energy, gradient, and Hessian-vector products
- **PETSc** distributed sparse linear algebra (CG + AMG preconditioner)
- **MPI-parallel** via DOF-based overlapping domain decomposition with P2P ghost exchange
- **Local graph coloring** — each rank colors its own subgraph (no broadcast)
- **vmap-batched HVP** — all colour directions computed in one JIT call

The code lives in [`pLaplace2D_jax_petsc/`](pLaplace2D_jax_petsc/), with the assembler in
[`parallel_hessian_dof.py`](pLaplace2D_jax_petsc/parallel_hessian_dof.py) and the Newton solver in
[`tools_petsc4py/minimizers.py`](tools_petsc4py/minimizers.py).

For detailed benchmark results see [`results_pLaplace.md`](results_pLaplace.md).  
For the technical description of the parallel decomposition see [`jax_parallel_partitioning.md`](jax_parallel_partitioning.md).

## 1. Imports and Configuration

The solver requires `mpi4py`, `petsc4py`, `jax`, `scipy`, and `igraph`.  
Thread pinning (`OMP_NUM_THREADS=1`) is critical for Hypre AMG to avoid oversubscription.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=false"

import numpy as np
import time
from mpi4py import MPI

from jax import config
config.update("jax_enable_x64", True)
config.update("jax_platform_name", "cpu")

from pLaplace2D_jax_petsc.mesh import MeshpLaplace2D
from pLaplace2D_jax_petsc.parallel_hessian_dof import LocalColoringAssembler
from tools_petsc4py.minimizers import newton

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
print(f"MPI rank {rank} of {comm.Get_size()}")

MPI rank 0 of 1


## 2. Load Mesh

Pre-generated P1 triangular meshes are stored as HDF5 files in `mesh_data/pLaplace/`.  
JAX level numbering is offset by +1 from FEniCS (e.g., JAX level 7 = FEniCS level 6 = 48 641 DOFs).

| JAX level | DOFs    | Elements |
|-----------|---------|----------|
| 5         | 2 945   | 5 632    |
| 7         | 48 641  | 95 744   |
| 9         | 784 385 | 1 564 672|

In [2]:
mesh_level = 7  # adjust: 5 (small), 7 (medium), 9 (large)

mesh_obj = MeshpLaplace2D(mesh_level=mesh_level)
params, adjacency, u_init = mesh_obj.get_data_jax()

n_dofs = len(u_init)
n_elems = params["elems"].shape[0]
print(f"Level {mesh_level}: {n_dofs:,} DOFs, {n_elems:,} elements")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Level 7: 48,641 DOFs, 98,304 elements


## 3. Build the Assembler

The `LocalColoringAssembler` handles everything:
1. **RCM reordering** of DOFs for spatial locality
2. **DOF partitioning** — each rank owns a contiguous block of reordered DOFs
3. **Local graph coloring** — each rank builds $A^2|_J$ from its local elements and colors with igraph
4. **JIT compilation** of energy, gradient, and batched HVP functions
5. **PETSc MPIAIJ matrix** setup with COO preallocation
6. **KSP solver** (CG + GAMG or Hypre BoomerAMG)

In [3]:
t0 = time.perf_counter()

assembler = LocalColoringAssembler(
    params=params,
    comm=comm,
    adjacency=adjacency,
    coloring_trials_per_rank=10,  # multi-start coloring heuristic
    ksp_rtol=1e-3,               # KSP relative tolerance
    ksp_type="cg",
    pc_type="gamg",              # or "hypre" for BoomerAMG
)

setup_time = time.perf_counter() - t0

part = assembler.part
print(f"Setup: {setup_time:.3f}s")
print(f"  Owned DOFs: [{part.lo}, {part.hi}) = {part.n_owned}")
print(f"  Local nodes: {part.n_local} (owned + ghost)")
print(f"  Local elements: {part.n_local_elems}")
print(f"  Graph colors: {assembler.n_colors}")
print(f"  Hessian NNZ (global): {assembler.nnz_global:,}")

Setup: 1.184s
  Owned DOFs: [0, 48641) = 48641
  Local nodes: 49663 (owned + ghost)
  Local elements: 98302
  Graph colors: 8
  Hessian NNZ (global): 338,449


## 4. Solve with Newton's Method

The Newton solver uses:
- **Golden-section line search** on $[-0.5, 2.0]$ (allows overshooting)
- **Convergence criteria**: $|J(u_{k+1}) - J(u_k)| < 10^{-5}$ or $\|\nabla J\|_2 < 10^{-3}$
- **Hessian assembly** via sparse finite differences: all $n_c$ colour HVPs computed in one `vmap` call, scattered into PETSc MPIAIJ via COO fast-path

In [4]:
# Prepare initial guess in reordered DOF space
u_init_reord = np.asarray(u_init, dtype=np.float64)[assembler.part.perm]
x = assembler.create_vec(u_init_reord)

# Solve
comm.Barrier()
t0 = time.perf_counter()

result = newton(
    energy_fn=assembler.energy_fn,
    gradient_fn=assembler.gradient_fn,
    hessian_solve_fn=assembler.hessian_solve_fn,
    x=x,
    tolf=1e-5,
    tolg=1e-3,
    linesearch_tol=1e-3,
    linesearch_interval=(-0.5, 2.0),
    maxit=100,
    verbose=True,
    comm=comm,
    save_history=True,
)

solve_time = time.perf_counter() - t0
print(f"\nSolve time: {solve_time:.3f}s")
print(f"Newton iterations: {result['nit']}")
print(f"Final energy J(u) = {result['fun']:.6f}")
print(f"Message: {result['message']}")

it=1, J=-0.49010, dJ=5.21145e+05, ||g||=2.80624e+04, alpha=1.99965e+00, ksp_its=6, ls_evals=17
it=2, J=-5.77522, dJ=5.28512e+00, ||g||=1.35502e-01, alpha=4.39357e-02, ksp_its=7, ls_evals=17
it=3, J=-7.78866, dJ=2.01344e+00, ||g||=4.75246e+00, alpha=6.74759e-01, ksp_its=6, ls_evals=17
it=4, J=-7.95708, dJ=1.68415e-01, ||g||=1.82704e+00, alpha=1.13790e+00, ksp_its=7, ls_evals=17
it=5, J=-7.95829, dJ=1.21284e-03, ||g||=1.43614e-01, alpha=1.03061e+00, ksp_its=6, ls_evals=17
it=6, J=-7.95829, dJ=1.87144e-06, ||g||=5.84248e-03, alpha=1.00931e+00, ksp_its=7, ls_evals=17

Solve time: 2.339s
Newton iterations: 6
Final energy J(u) = -7.958292
Message: Energy change converged


## 5. Timing Breakdown

Analyse where time is spent in each Newton iteration.

In [5]:
history = result.get("history", [])
if history:
    print(f"{'It':>3s} {'grad':>8s} {'hess':>8s} {'LS':>8s} {'KSP its':>8s} {'LS evals':>8s}")
    print("-" * 48)
    for h in history:
        print(f"{h['it']:3d} {h['t_grad']:8.4f} {h['t_hess']:8.4f} "
              f"{h['t_ls']:8.4f} {h['ksp_its']:8d} {h['ls_evals']:8d}")
    print("-" * 48)
    t_grad = sum(h['t_grad'] for h in history)
    t_hess = sum(h['t_hess'] for h in history)
    t_ls = sum(h['t_ls'] for h in history)
    total_ksp = sum(h['ksp_its'] for h in history)
    total_ls = sum(h['ls_evals'] for h in history)
    print(f"{'SUM':>3s} {t_grad:8.4f} {t_hess:8.4f} {t_ls:8.4f} {total_ksp:8d} {total_ls:8d}")
    print(f"\nTotal: grad={t_grad:.3f}s + hess={t_hess:.3f}s + LS={t_ls:.3f}s = {t_grad+t_hess+t_ls:.3f}s")

 It     grad     hess       LS  KSP its LS evals
------------------------------------------------
  1   0.1101   0.3888   0.0206        6       17
  2   0.0043   0.3360   0.0218        7       17
  3   0.0044   0.3021   0.0196        6       17
  4   0.0068   0.3341   0.0200        7       17
  5   0.0047   0.3091   0.0224        6       17
  6   0.0076   0.3217   0.0209        7       17
------------------------------------------------
SUM   0.1378   1.9918   0.1254       39      102

Total: grad=0.138s + hess=1.992s + LS=0.125s = 2.255s


## 6. Assembly Details

Per-iteration Hessian assembly breakdown: P2P ghost exchange, batched HVP, COO insertion, and KSP solve.

In [6]:
if assembler.iter_timings:
    print(f"{'It':>3s} {'P2P':>8s} {'HVP':>8s} {'COO':>8s} {'KSP':>8s} {'KSP its':>8s}")
    print("-" * 48)
    for i, d in enumerate(assembler.iter_timings):
        print(f"{i:3d} {d.get('p2p_exchange', d.get('allgatherv', 0)):8.4f} "
              f"{d.get('hvp_compute', 0):8.4f} "
              f"{d.get('coo_assembly', 0):8.4f} "
              f"{d.get('ksp', 0):8.4f} {d.get('ksp_its', 0):8d}")

 It      P2P      HVP      COO      KSP  KSP its
------------------------------------------------
  0   0.0004   0.0074   0.0028   0.3701        6
  1   0.0003   0.0062   0.0005   0.3241        7
  2   0.0004   0.0070   0.0010   0.2858        6
  3   0.0004   0.0086   0.0005   0.3177        7
  4   0.0003   0.0070   0.0010   0.2939        6
  5   0.0003   0.0045   0.0005   0.3077        7


## 7. Cleanup

In [7]:
x.destroy()
assembler.cleanup()
print("PETSc objects destroyed.")

PETSc objects destroyed.


## Running with MPI

This notebook runs in serial (1 rank) when executed directly in Jupyter.  
For **parallel execution**, use the CLI script:

```bash
# Serial
mpirun -n 1 python3 -m pLaplace2D_jax_petsc.solve_pLaplace_dof --local-coloring --level 7

# 4 ranks
mpirun -n 4 python3 -m pLaplace2D_jax_petsc.solve_pLaplace_dof --local-coloring --level 9

# 16 ranks, Hypre AMG
mpirun -n 16 python3 -m pLaplace2D_jax_petsc.solve_pLaplace_dof --local-coloring --level 9 --pc-type hypre
```

Or via Docker:
```bash
docker run --rm -v "$PWD":/workspace -w /workspace fenics_test:latest \
  mpirun -n 16 python3 -m pLaplace2D_jax_petsc.solve_pLaplace_dof --local-coloring --level 9
```